# Análisis Biomecánico del Curl de Bíceps con MediaPipe
Este notebook implementa un sistema de evaluación en tiempo real del ejercicio de Curl de Bíceps utilizando estimación de pose (MediaPipe). Se calculan coordenadas 3D, distancias, ángulos articulares, orientación de segmentos y cinemática muscular para analizar la técnica y detectar errores.

## 1. Importación de Librerías Necesarias
Importamos las librerías requeridas para el procesamiento de video, cálculos matemáticos y visualización.

In [3]:
# Librerías para procesamiento de video, pose estimation y visualización
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from math import atan2, degrees, acos, sqrt, pi

## 2. Carga del Modelo de Pose Estimation (MediaPipe)
Inicializamos el modelo de MediaPipe Pose para detectar los landmarks corporales en cada frame del video.

In [4]:
# 1. Verifica la versión y ruta de MediaPipe activa
def info_mediapipe():
    import mediapipe as mp
    print('Versión:', mp.__version__)
    print('Ruta:', mp.__file__)
info_mediapipe()

# 2. Desinstala e instala la versión clásica y estable de MediaPipe
# Ejecuta estas líneas en una celda separada si sigues teniendo problemas:
# %pip uninstall -y mediapipe
# %pip install mediapipe==0.10.0

# 3. Reinicia el kernel tras la instalación y ejecuta:
# import mediapipe as mp
# mp_pose = mp.solutions.pose
# mp_drawing = mp.solutions.drawing_utils
# pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)

Versión: 0.10.33
Ruta: c:\Users\Eric\OneDrive - Universidad de Castilla-La Mancha\Documentos\GitHub\Desarrollo-de-una-herramienta-de-evaluaci-n-biomec-nica-en-tiempo-real-con-MediaPipe\.entorno\Lib\site-packages\mediapipe\__init__.py


## 3. Extracción de Coordenadas de Landmarks del Tren Superior
Procesamos el video para extraer las coordenadas (x, y, z) de los hombros (11, 12), codos (13, 14) y muñecas (15, 16).

In [5]:
# Función para extraer coordenadas de los landmarks del tren superior
def extraer_landmarks(frame, pose):
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)
    if not results.pose_landmarks:
        return None
    # Landmarks: 11-hombro izq, 12-hombro der, 13-codo izq, 14-codo der, 15-muñeca izq, 16-muñeca der
    puntos = {}
    for idx in [11, 12, 13, 14, 15, 16]:
        lm = results.pose_landmarks.landmark[idx]
        puntos[idx] = (lm.x, lm.y, lm.z)
    return puntos

# Ejemplo de uso con un frame de video
# cap = cv2.VideoCapture('ruta_al_video.mp4')
# ret, frame = cap.read()
# puntos = extraer_landmarks(frame, pose)
# print(puntos)

## 4. Cálculo de Distancias Euclidiana (Longitud de Húmero y Antebrazo)
Calculamos la longitud del húmero (hombro-codo) y antebrazo (codo-muñeca) usando la distancia Euclidiana entre los puntos articulares.

In [6]:
# Cálculo de distancia euclidiana entre dos puntos 3D
def distancia_3d(p1, p2):
    return sqrt((p2[0]-p1[0])**2 + (p2[1]-p1[1])**2 + (p2[2]-p1[2])**2)

# Ejemplo: longitud del húmero y antebrazo (lado izquierdo)
def longitudes_segmentos(puntos):
    humero = distancia_3d(puntos[11], puntos[13])  # hombro-codo
    antebrazo = distancia_3d(puntos[13], puntos[15])  # codo-muñeca
    return humero, antebrazo

# # Uso:
# humero, antebrazo = longitudes_segmentos(puntos)
# print('Longitud húmero:', humero, 'Longitud antebrazo:', antebrazo)

In [ ]:
# Captura de video por webcam, detección de puntos, cálculo y visualización en tiempo real
import cv2
import mediapipe as mp
import numpy as np
from math import atan2, degrees, acos, sqrt, pi

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)
    if results.pose_landmarks:
        puntos = {}
        for idx in [11, 12, 13, 14, 15, 16]:
            lm = results.pose_landmarks.landmark[idx]
            h, w, _ = frame.shape
            puntos[idx] = (lm.x * w, lm.y * h, lm.z * w)
        # Cálculo de ángulo de codo
        try:
            ang3d, ang2d = angulo_codo(puntos)
            cv2.putText(frame, f'Angulo codo: {ang3d:.1f}°', (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
        except Exception:
            pass
        # Dibuja los landmarks
        mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
    cv2.imshow('Curl Biceps Biomecanica', frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC para salir
        break
cap.release()
cv2.destroyAllWindows()

## 5. Cálculo del Ángulo Articular del Codo (Producto Escalar y Arcotangente)
Calculamos el ángulo de flexión del codo usando el producto escalar y la función arctan2, considerando el hombro como punto A, el codo como vértice B y la muñeca como punto C.

In [7]:
# Cálculo del ángulo del codo (flexión) usando producto escalar y arctan2

def angulo_codo(puntos):
    # Vector BA (hombro-codo), Vector BC (muñeca-codo)
    A = np.array(puntos[11])
    B = np.array(puntos[13])
    C = np.array(puntos[15])
    BA = A - B
    BC = C - B
    # Producto escalar
    cos_theta = np.dot(BA, BC) / (np.linalg.norm(BA) * np.linalg.norm(BC))
    angulo_rad = np.arccos(np.clip(cos_theta, -1.0, 1.0))
    angulo_deg = np.degrees(angulo_rad)
    # Alternativa 2D con arctan2
    theta_2d = atan2(C[1]-B[1], C[0]-B[0]) - atan2(A[1]-B[1], A[0]-B[0])
    theta_2d = np.abs(np.degrees(theta_2d))
    return angulo_deg, theta_2d

# # Uso:
# ang3d, ang2d = angulo_codo(puntos)
# print('Ángulo codo 3D:', ang3d, 'Ángulo codo 2D:', ang2d)

## 6. Evaluación de la Verticalidad del Húmero (Ángulo de Inclinación)
Calculamos el ángulo de inclinación del húmero respecto a la vertical usando arctan2, y verificamos que se mantenga entre 0° y 10° para una forma estricta.

In [8]:
# Ángulo de inclinación del húmero respecto a la vertical (eje Y)
def inclinacion_humero(puntos):
    hombro = puntos[11]
    codo = puntos[13]
    dx = codo[0] - hombro[0]
    dy = codo[1] - hombro[1]
    angulo_rad = atan2(dy, dx)
    angulo_deg = np.abs(degrees(angulo_rad))
    return angulo_deg

# # Uso:
# ang_humero = inclinacion_humero(puntos)
# print('Inclinación húmero:', ang_humero)

## 7. Detección de 'Cheating' (Flexión Excesiva del Hombro)
Identificamos cuando la flexión del hombro supera los 15°, indicando el uso de estrategias compensatorias que reducen la activación del bíceps.

In [9]:
# Detección de cheating: flexión del hombro mayor a 15°
def detectar_cheating(angulo_humero):
    if angulo_humero > 15:
        return True  # Hay cheating
    elif angulo_humero > 10:
        return 'Advertencia'  # Técnica no estricta
    else:
        return False  # Forma estricta

# # Uso:
# cheat = detectar_cheating(ang_humero)
# print('Cheating:', cheat)

## 8. Cálculo de Cinemática (Velocidad y Fase Ascendente/Descendente)
Calculamos la velocidad y posición en el tiempo para dividir el ejercicio en fase concéntrica (ascendente) y excéntrica (descendente).

In [10]:
# Cálculo de velocidad angular y fases del movimiento

def calcular_fases(angulos, tiempos):
    velocidades = np.gradient(angulos, tiempos)
    fases = []
    for v in velocidades:
        if v > 0:
            fases.append('Ascendente')  # Concéntrica
        else:
            fases.append('Descendente')  # Excéntrica
    return velocidades, fases

# # Uso:
# angulos = [...]  # lista de ángulos de codo por frame
# tiempos = [...]  # lista de tiempos por frame
# velocidades, fases = calcular_fases(angulos, tiempos)
# print(fases)

## 9. Análisis del Tipo de Agarre (Supino vs Pronado)
Determinamos el tipo de agarre basado en la orientación de las muñecas para evaluar la excitación muscular del bíceps braquial y deltoides anterior.

In [11]:
# Análisis del tipo de agarre según la orientación de la muñeca
# Supinación: palma hacia arriba, pronación: palma hacia abajo

def tipo_agarre(puntos):
    # Usamos la diferencia entre el eje Y de la muñeca y el codo
    codo = puntos[13]
    muneca = puntos[15]
    # Si la muñeca está por debajo del codo en Y, es supino (palma arriba)
    if muneca[1] > codo[1]:
        return 'Supino (palma arriba)'
    else:
        return 'Prono/Neutro (palma abajo o neutro)'

# # Uso:
# agarre = tipo_agarre(puntos)
# print('Tipo de agarre:', agarre)

## 10. Visualización de Resultados y Métricas Biomecánicas
Creamos visualizaciones de los ángulos calculados, fases del ejercicio y métricas de técnica para evaluar la calidad del movimiento.

In [12]:
# Visualización de resultados: ángulos, fases y técnica

def visualizar_resultados(tiempos, angulos, fases, inclinaciones, cheating):
    plt.figure(figsize=(12,6))
    plt.subplot(2,1,1)
    plt.plot(tiempos, angulos, label='Ángulo de codo (°)')
    plt.plot(tiempos, inclinaciones, label='Inclinación húmero (°)')
    plt.legend()
    plt.ylabel('Ángulo (°)')
    plt.title('Ángulos articulares y verticalidad')
    plt.subplot(2,1,2)
    plt.plot(tiempos, fases, label='Fase', color='orange')
    plt.plot(tiempos, cheating, label='Cheating', color='red', linestyle='dashed')
    plt.ylabel('Fase/Cheating')
    plt.xlabel('Tiempo (s)')
    plt.legend()
    plt.tight_layout()
    plt.show()

# # Uso:
# visualizar_resultados(tiempos, angulos, fases, inclinaciones, cheating)